In [ ]:
import pandas as pd
import pickle
import os
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.decomposition import IncrementalPCA

In [ ]:
def product_vectors_incremental():
    csv_path = os.path.join("..","..","Task 1", "mongodb", "product_vec.parquet")
    df = pd.read_parquet(csv_path)
    
    #increase features/components if MRR is bad
    hasher = HashingVectorizer(n_features=1000, alternate_sign=False)
    ipca = IncrementalPCA(n_components=50)

    text_data = df['Small'].astype(str).fillna('')
    X_hashed = hasher.transform(text_data)
    X_pca = ipca.fit_transform(X_hashed.toarray())

    #rename parent_asin to _id as problems occured with MRR
    output_df = pd.DataFrame({
        '_id': df['parent_asin'].astype(str), 
        'Small': df['Small'],
        'vector': X_pca.tolist()
    })
    
    #create a pkl file since MRR gave 0.0000 otherwise
    output_df.to_parquet("product_vectors_incremental_pca.parquet")
    with open("pca_model.pkl", "wb") as f:
        pickle.dump(ipca, f)

    print(f"Success! Saved {len(output_df)} rows.")

if __name__ == "__main__":
    product_vectors_incremental()

In [ ]:
def user_vectors_incremental():
    csv_path = os.path.join("..","..","Task 1", "mongodb", "user_vec.parquet")
    df = pd.read_parquet(csv_path)
    
    #increase features/components if MRR is bad
    hasher = HashingVectorizer(n_features=1000, alternate_sign=False)
    ipca = IncrementalPCA(n_components=50)

    text_data = df['item_string'].astype(str).fillna('')
    X_hashed = hasher.transform(text_data)
    X_pca = ipca.fit_transform(X_hashed.toarray())

    #rename parent_asin to _id as problems occured with MRR
    output_df = pd.DataFrame({
        '_id': df['_id'].astype(str), 
        'metadata': [{'user_id': str(uid)} for uid in df['_id']],
        'vector': X_pca.tolist()
    })
    
    #create a pkl file since MRR gave 0.0000 otherwise
    output_df.to_parquet("user_vectors_incremental_pca.parquet")
    with open("pca_model_user.pkl", "wb") as f:
        pickle.dump(ipca, f)

    print(f"Success! Saved {len(output_df)} rows.")

if __name__ == "__main__":
    product_vectors_incremental()